# Arrow Tables vs Datasets

In [ ]:
! pip install pyarrow
! pip install pandas

## Imports

In [1]:
import pyarrow.csv as csv

## Charger une table en mémoire avec Arrow DataFrame

In [2]:
csv_file = "data1.csv"
table = csv.read_csv(csv_file)

# Trier les données par une colonne spécifique
sorted_table = table.sort_by("name")
print(sorted_table)

pyarrow.Table
name: string
phone: string
email: string
postalZip: string
Salary: string
country: string
age: int64
user_id: int64
----
name: [["Abbot Chang","Adara Sanchez","Aileen O'Neill","Aretha Chen","Ashton Castaneda",...,"Walter Wiggins","Whilemina Nielsen","William Morse","Winter Luna","Wyoming Kelly"]]
phone: [["06 40 44 38 71","01 66 30 28 05","05 87 31 31 53","09 56 23 98 48","03 44 66 36 63",...,"08 04 23 36 41","01 06 54 85 38","03 23 95 87 15","09 84 18 87 87","06 28 66 68 61"]]
email: [["vitae@hotmail.edu","metus.vitae@google.net","cursus.et@hotmail.ca","cum.sociis.natoque@hotmail.edu","vehicula@icloud.edu",...,"mollis.non@icloud.org","curabitur@google.net","magna.phasellus@hotmail.couk","mauris.vestibulum@google.edu","cursus.nunc.mauris@outlook.net"]]
postalZip: [["18986","75641","U1V 1IS","3667","37215",...,"587379","5273 UF","46315","5887","2731"]]
Salary: [["$5,689","$7,095","$8,079","$7,624","$5,068",...,"$5,948","$6,524","$6,032","$5,854","$6,466"]]
country: [["Germ

## Travailler sur un dataset qui dépasse la mémoire

In [3]:
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd

# Charger un dataset Parquet
#parquet_file = "Path/to/parquet"
parquet_file = r"C:\Users\afeldmann\Desktop\openamir"
dataset = ds.dataset(parquet_file, format="parquet", partitioning="hive")

# Appliquer des filtres et des agrégations
filtered = dataset.filter(pc.field("BEN_SEX_COD") == "1")

### C'est lazy !

In [4]:
filtered.count_rows()

1301140328

In [5]:
filtered.head(20)

pyarrow.Table
FLX_ANN_MOI: string
ORG_CLE_REG: string
AGE_BEN_SNDS: string
BEN_RES_REG: string
BEN_CMU_TOP: string
BEN_QLT_COD: string
BEN_SEX_COD: string
DDP_SPE_COD: string
ETE_CAT_SNDS: string
ETE_REG_COD: string
ETE_TYP_SNDS: string
ETP_REG_COD: string
ETP_CAT_SNDS: string
MDT_TYP_COD: string
MFT_COD: string
PRS_FJH_TYP: string
PRS_ACT_COG: double
PRS_ACT_NBR: double
PRS_ACT_QTE: double
PRS_DEP_MNT: double
PRS_PAI_MNT: double
PRS_REM_BSE: double
PRS_REM_MNT: double
FLT_ACT_COG: double
FLT_ACT_NBR: double
FLT_ACT_QTE: double
FLT_PAI_MNT: double
FLT_DEP_MNT: double
FLT_REM_MNT: double
SOI_ANN: int32
SOI_MOI: int32
ASU_NAT: string
ATT_NAT: string
CPL_COD: string
CPT_ENV_TYP: string
DRG_AFF_NAT: string
ETE_IND_TAA: string
EXO_MTF: string
MTM_NAT: string
PRS_NAT: string
PRS_PPU_SEC: string
PRS_REM_TAU: double
PRS_REM_TYP: string
PRS_PDS_QCP: string
EXE_INS_REG: string
PSE_ACT_SNDS: string
PSE_ACT_CAT: string
PSE_SPE_SNDS: string
PSE_STJ_SNDS: string
PRE_INS_REG: string
PSP_ACT_SNDS: str

### Maintenant une aggregation avec Acero :

In [6]:
import pyarrow.dataset as ds
import pyarrow.acero as acero
import pyarrow.compute as pc

emplacement_dataset = r"C:\Users\afeldmann\Desktop\openamir"

dataset = ds.dataset(
    emplacement_dataset,
    format="parquet",
    partitioning="hive"
)

In [7]:
plan = acero.Declaration(
    "scan",
    acero.ScanNodeOptions(dataset, columns=["FLT_PAI_MNT", "annee", "mois"])
)

plan = acero.Declaration(
    "project",
    acero.ProjectNodeOptions(
        expressions=[
            pc.field("FLT_PAI_MNT"),
            pc.field("annee"),
            pc.field("mois"),
        ],
        names=["FLT_PAI_MNT", "annee", "mois"]
    ),
    inputs=[plan]
)

plan = acero.Declaration(
    "aggregate",
    acero.AggregateNodeOptions(
        aggregates=[
            ("FLT_PAI_MNT", "hash_sum", pc.ScalarAggregateOptions(skip_nulls=False, min_count=0), "FLT_PAI_MNT")
        ],
        keys=["annee", "mois"]
    ),
    inputs=[plan]
)


In [8]:
plan

<pyarrow.acero.Declaration>
ExecPlan with 4 nodes:
:ConsumingSinkNode{}
  :GroupByNode{keys=["annee", "mois"], aggregates=[
  	hash_sum(FLT_PAI_MNT, {skip_nulls=false, min_count=0}),
  ]}
    :ProjectNode{projection=[FLT_PAI_MNT, annee, mois]}
      :SourceNode{}

In [9]:
table = plan.to_table()
print(table)

pyarrow.Table
annee: int32
mois: int32
FLT_PAI_MNT: double
----
annee: [[2018,2018,2018,2018,2018,...,2024,2024,2024,2024,2024]]
mois: [[1,2,3,4,5,...,8,9,10,11,12]]
FLT_PAI_MNT: [[1.1449685652927162e+10,1.0650381771494055e+10,1.1552585152688412e+10,1.1526760891298973e+10,1.1030467262339811e+10,...,1.107497530681342e+10,1.3362059430379034e+10,1.5555087741627312e+10,1.3508940256071648e+10,1.4659206975162193e+10]]


## Retourner en pandas pour travailler les résultats simplement :

In [10]:
# Convertir en DataFrame Pandas
pandas_df = table.to_pandas()
print(pandas_df)

    annee  mois   FLT_PAI_MNT
0    2018     1  1.144969e+10
1    2018     2  1.065038e+10
2    2018     3  1.155259e+10
3    2018     4  1.152676e+10
4    2018     5  1.103047e+10
..    ...   ...           ...
79   2024     8  1.107498e+10
80   2024     9  1.336206e+10
81   2024    10  1.555509e+10
82   2024    11  1.350894e+10
83   2024    12  1.465921e+10

[84 rows x 3 columns]
